# Similarity

Compares frozen sentence-embedding models across every edit-generality, edit-locality, and generality-locality prompt pair available within each UniEdit record. The analysis is independent of the edited language model.

## Environment And Paths

In [ ]:
from __future__ import annotations

from contextlib import redirect_stderr
import gc
from io import StringIO
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_REPO = "Savassan/KnowledgeSurgeon"
DATASET_CONFIG = "similarity"
DATASET_SPLIT = "data"
OUTPUT_DIR = PROJECT_ROOT / "data" / "similarity" / "embedding"
REPRESENTATION_TYPE = "normalized_dense_sentence_embedding"

EMBEDDING_MODEL_SPECS = {
    "all-mpnet-base-v2": {
        "repo": "sentence-transformers/all-mpnet-base-v2",
        "backend": "sentence_transformers",
        "input_prefix": "",
    },
    "all-MiniLM-L6-v2": {
        "repo": "sentence-transformers/all-MiniLM-L6-v2",
        "backend": "sentence_transformers",
        "input_prefix": "",
    },
    "all-MiniLM-L12-v2": {
        "repo": "sentence-transformers/all-MiniLM-L12-v2",
        "backend": "sentence_transformers",
        "input_prefix": "",
    },
    "e5-base-v2": {
        "repo": "intfloat/e5-base-v2",
        "backend": "sentence_transformers",
        "input_prefix": "query: ",
    },
    "e5-large-v2": {
        "repo": "intfloat/e5-large-v2",
        "backend": "sentence_transformers",
        "input_prefix": "query: ",
    },
    "bge-base-en-v1.5": {
        "repo": "BAAI/bge-base-en-v1.5",
        "backend": "sentence_transformers",
        "input_prefix": "",
    },
    "bge-large-en-v1.5": {
        "repo": "BAAI/bge-large-en-v1.5",
        "backend": "sentence_transformers",
        "input_prefix": "",
    },
    "bge-m3": {
        "repo": "BAAI/bge-m3",
        "backend": "flag_embedding",
        "input_prefix": "",
    },
}


def resolve_embedding_model_keys(model_keys: list[str]) -> list[str]:
    selected = list(dict.fromkeys(model_keys))
    if not selected:
        raise ValueError("embedding_model_keys must contain at least one model.")
    unknown = [key for key in selected if key not in EMBEDDING_MODEL_SPECS]
    if unknown:
        raise ValueError(f"Unknown embedding model keys: {unknown}")
    return selected


def similarity_output_path(model_key: str, record_limit: int | None) -> Path:
    if record_limit is not None and int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    output_stem = model_key if record_limit is None else f"{model_key}.limit-{int(record_limit)}"
    return OUTPUT_DIR / embedding_precision_mode() / f"{output_stem}.json"


print(f"output_dir={OUTPUT_DIR}")

## Embedding Backends

In [ ]:
def embedding_device() -> str:
    return f"cuda:{EMBEDDING_GPU_ID}" if torch.cuda.is_available() else "cpu"


def embedding_compute_dtype() -> str:
    return "float16" if EMBEDDING_USE_FP16 and torch.cuda.is_available() else "float32"


def embedding_precision_mode() -> str:
    return "fp16" if embedding_compute_dtype() == "float16" else "fp32"


def load_embedding_model(model_key: str):
    spec = EMBEDDING_MODEL_SPECS[model_key]
    device = embedding_device()
    if spec["backend"] == "sentence_transformers":
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(spec["repo"], device=device)
        model.max_seq_length = min(int(model.max_seq_length), EMBEDDING_MAX_LENGTH)
        if embedding_compute_dtype() == "float16":
            model.half()
    elif spec["backend"] == "flag_embedding":
        from FlagEmbedding import BGEM3FlagModel

        model = BGEM3FlagModel(
            spec["repo"],
            use_fp16=embedding_compute_dtype() == "float16",
            devices=device,
        )
    else:
        raise ValueError(f"Unsupported embedding backend: {spec['backend']}")
    print(
        f"loaded_embedding_model={model_key} repo={spec['repo']} "
        f"backend={spec['backend']} device={device} "
        f"compute_dtype={embedding_compute_dtype()}"
    )
    return model


def embedding_effective_max_length(model, model_key: str) -> int:
    if EMBEDDING_MODEL_SPECS[model_key]["backend"] == "sentence_transformers":
        return int(model.max_seq_length)
    return EMBEDDING_MAX_LENGTH


def format_embedding_prompts(model_key: str, prompts: list[str]) -> list[str]:
    prefix = EMBEDDING_MODEL_SPECS[model_key]["input_prefix"]
    return [f"{prefix}{prompt}" for prompt in prompts]


def encode_flag_embedding_prompts(model, prompts: list[str]) -> dict:
    with redirect_stderr(StringIO()):
        return model.encode(
            prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            max_length=EMBEDDING_MAX_LENGTH,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )


def encode_prompts(model, model_key: str, prompts: list[str]) -> np.ndarray:
    spec = EMBEDDING_MODEL_SPECS[model_key]
    formatted_prompts = format_embedding_prompts(model_key, prompts)
    if spec["backend"] == "sentence_transformers":
        embeddings = model.encode(
            formatted_prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    else:
        encoded = encode_flag_embedding_prompts(model, formatted_prompts)
        embeddings = encoded["dense_vecs"]

    embeddings = np.asarray(embeddings, dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=-1, keepdims=True)
    embeddings = embeddings / np.clip(norms, 1e-12, None)
    if embeddings.ndim != 2 or embeddings.shape[0] != len(prompts):
        raise ValueError(
            f"Unexpected embedding shape for {model_key}: {embeddings.shape}; "
            f"expected ({len(prompts)}, hidden_size)."
        )
    return embeddings


def move_embedding_model_to_cpu(model, model_key: str) -> None:
    backend = EMBEDDING_MODEL_SPECS[model_key]["backend"]
    try:
        if backend == "sentence_transformers":
            model.to("cpu")
        else:
            underlying_model = getattr(model, "model", None)
            if underlying_model is not None and hasattr(underlying_model, "to"):
                underlying_model.to("cpu")
    except Exception as error:
        print(f"embedding_model_cpu_move_warning={model_key}: {error}")


def clear_embedding_memory(model_key: str) -> None:
    gc.collect()
    if not torch.cuda.is_available():
        return
    try:
        torch.cuda.synchronize(EMBEDDING_GPU_ID)
    except Exception:
        pass
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    allocated = torch.cuda.memory_allocated(EMBEDDING_GPU_ID) / 1024**3
    reserved = torch.cuda.memory_reserved(EMBEDDING_GPU_ID) / 1024**3
    print(
        f"released_embedding_model={model_key} "
        f"cuda_allocated_gb={allocated:.3f} cuda_reserved_gb={reserved:.3f}"
    )

## Similarity Records

Only prompt text is embedded. Each result retains source indices and pair values; prompts and targets remain available from the source dataset without being duplicated into every model result.

In [ ]:
PAIR_AGGREGATION = "all_prompt_pairs"
PAIR_SPECS = (
    ("p1-p2", "edit", "generality", "edit - generality"),
    ("p1-p3", "edit", "locality", "edit - locality"),
    ("p2-p3", "generality", "locality", "generality - locality"),
)


def build_similarity_records(dataset, record_limit: int | None) -> list[dict]:
    source_count = len(dataset) if record_limit is None else min(int(record_limit), len(dataset))
    records = []
    for source_index in range(source_count):
        source_record = dataset[source_index]
        edit_prompt = source_record["edit"]["prompt"]
        generality_prompts = list(source_record["generality"])
        locality_prompts = [item["prompt"] for item in source_record["locality"]]
        role_prompts = {
            "edit": [edit_prompt],
            "generality": generality_prompts,
            "locality": locality_prompts,
        }
        for role, prompts in role_prompts.items():
            if not prompts or any(not prompt for prompt in prompts):
                raise ValueError(
                    f"Empty {role} prompt list at source index {source_index}."
                )
        records.append({
            "source_index": source_index,
            "role_prompts": role_prompts,
        })
    return records

## Sentence-Embedding Analysis

Sentence Transformers models use their checkpoint-defined pooling and normalization. BGE-M3 uses FlagEmbedding's dense output. All outputs are normalized again before cosine comparison for a consistent experiment.

In [ ]:
def analyze_record_batch(
    record_batch: list[dict],
    model,
    model_key: str,
) -> list[dict]:
    prompts = [
        prompt
        for record in record_batch
        for role_prompts in record["role_prompts"].values()
        for prompt in role_prompts
    ]
    embeddings = encode_prompts(model, model_key, prompts)

    analyzed_records = []
    offset = 0
    for record in record_batch:
        role_embeddings = {}
        prompt_counts = {}
        for role, role_prompts in record["role_prompts"].items():
            count = len(role_prompts)
            role_embeddings[role] = embeddings[offset : offset + count]
            prompt_counts[role] = count
            offset += count

        pair_values = {}
        for pair_key, left_role, right_role, _ in PAIR_SPECS:
            matrix = role_embeddings[left_role] @ role_embeddings[right_role].T
            pair_values[pair_key] = [
                round(float(value), 8) for value in matrix.reshape(-1)
            ]
        analyzed_records.append({
            "source_index": record["source_index"],
            "prompt_counts": prompt_counts,
            "pair_values": pair_values,
        })
    return analyzed_records


def aggregate_records(records: list[dict]) -> dict:
    if not records:
        raise ValueError("No records were analyzed.")

    pairs = {}
    for pair_key, _, _, label in PAIR_SPECS:
        values = np.asarray(
            [
                value
                for record in records
                for value in record["pair_values"][pair_key]
            ],
            dtype=np.float64,
        )
        if values.size == 0:
            raise ValueError(f"No values were produced for pair {pair_key}.")
        pairs[pair_key] = {
            "label": label,
            "count": int(values.size),
            "mean": round(float(values.mean()), 8),
            "std": round(float(values.std(ddof=0)), 8),
        }
    return {
        "record_count": len(records),
        "pairs": pairs,
    }


def write_json_atomic(payload: dict, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(f"{destination.suffix}.tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=True, allow_nan=False)
    temporary.replace(destination)


def similarity_dataset_metadata(dataset) -> dict:
    return {
        "repo": DATASET_REPO,
        "config": DATASET_CONFIG,
        "split": DATASET_SPLIT,
        "rows": len(dataset),
        "fingerprint": getattr(dataset, "_fingerprint", None),
    }


def run_model_similarity_analysis(
    dataset,
    records: list[dict],
    model,
    model_key: str,
    output_path: Path,
    record_batch_size: int,
) -> None:
    if record_batch_size <= 0:
        raise ValueError("record_batch_size must be positive.")

    completed_records = []
    for start in tqdm(
        range(0, len(records), record_batch_size),
        desc=f"Analyze embeddings [{model_key}]",
    ):
        completed_records.extend(
            analyze_record_batch(
                records[start : start + record_batch_size],
                model=model,
                model_key=model_key,
            )
        )

    completed_records.sort(key=lambda record: record["source_index"])
    spec = EMBEDDING_MODEL_SPECS[model_key]
    result = {
        "embedding_model_key": model_key,
        "embedding_model": spec["repo"],
        "embedding_backend": spec["backend"],
        "input_prefix": spec["input_prefix"],
        "representation": REPRESENTATION_TYPE,
        "use_fp16": EMBEDDING_USE_FP16,
        "compute_dtype": embedding_compute_dtype(),
        "max_length_cap": EMBEDDING_MAX_LENGTH,
        "effective_max_length": embedding_effective_max_length(model, model_key),
        "dataset": similarity_dataset_metadata(dataset),
        "pair_aggregation": PAIR_AGGREGATION,
        "pair_order": [
            {"pair": pair_key, "label": label}
            for pair_key, _, _, label in PAIR_SPECS
        ],
        "records": completed_records,
        "aggregate": aggregate_records(completed_records),
    }
    write_json_atomic(result, output_path)
    print(f"saved={output_path} records={len(completed_records)}")

In [ ]:
def result_matches_configuration(
    output_path: Path,
    model_key: str,
    dataset_metadata: dict | None = None,
) -> bool:
    if not output_path.exists():
        return False
    with open(output_path, encoding="utf-8") as file:
        result = json.load(file)
    spec = EMBEDDING_MODEL_SPECS[model_key]
    return (
        result.get("embedding_model_key") == model_key
        and result.get("embedding_model") == spec["repo"]
        and result.get("embedding_backend") == spec["backend"]
        and result.get("input_prefix") == spec["input_prefix"]
        and result.get("representation") == REPRESENTATION_TYPE
        and result.get("use_fp16") == EMBEDDING_USE_FP16
        and result.get("compute_dtype") == embedding_compute_dtype()
        and result.get("max_length_cap") == EMBEDDING_MAX_LENGTH
        and result.get("pair_aggregation") == PAIR_AGGREGATION
        and (dataset_metadata is None or result.get("dataset") == dataset_metadata)
        and result.get("pair_order") == [
            {"pair": pair_key, "label": label}
            for pair_key, _, _, label in PAIR_SPECS
        ]
    )


def calculate_similarity(
    embedding_model_keys: list[str],
    record_limit: int | None,
    record_batch_size: int,
    overwrite_output: bool,
) -> None:
    model_keys = resolve_embedding_model_keys(embedding_model_keys)
    dataset = load_dataset(DATASET_REPO, DATASET_CONFIG, split=DATASET_SPLIT)
    dataset_metadata = similarity_dataset_metadata(dataset)
    calculation_keys = []
    for model_key in model_keys:
        output_path = similarity_output_path(model_key, record_limit)
        if output_path.exists() and overwrite_output:
            output_path.unlink()
            print(f"removed_for_overwrite={output_path}")
        elif output_path.exists():
            if result_matches_configuration(output_path, model_key, dataset_metadata):
                print(f"skipped_existing={output_path}")
                continue
            print(f"recalculating_stale={output_path}")
        calculation_keys.append(model_key)
    if not calculation_keys:
        dataset = None
        gc.collect()
        return

    records = build_similarity_records(dataset, record_limit=record_limit)
    print(f"records={len(records)} models={calculation_keys}")
    failures = []
    for model_key in calculation_keys:
        model = None
        try:
            model = load_embedding_model(model_key)
            run_model_similarity_analysis(
                dataset,
                records,
                model=model,
                model_key=model_key,
                output_path=similarity_output_path(model_key, record_limit),
                record_batch_size=record_batch_size,
            )
        except Exception as error:
            failures.append(f"{model_key}: {type(error).__name__}: {error}")
            print(f"embedding_model_failed={failures[-1]}")
        finally:
            if model is not None:
                move_embedding_model_to_cpu(model, model_key)
            model = None
            clear_embedding_memory(model_key)

    dataset = None
    records = None
    gc.collect()
    if failures:
        raise RuntimeError("Embedding models failed:\n" + "\n".join(failures))

## Aggregate Tables And Plot

In [ ]:
PAIR_COLORS = {
    "p1-p2": "#0072B2",
    "p1-p3": "#E69F00",
    "p2-p3": "#009E73",
}


def aggregate_pair_table(results: list[tuple[str, dict]], show_std: bool) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        row = {"Embedding model": model_key}
        for pair_key, _, _, label in PAIR_SPECS:
            pair = result["aggregate"]["pairs"][pair_key]
            row[label] = (
                f"{pair['mean']:.4f} +/- {pair['std']:.4f}"
                if show_std
                else f"{pair['mean']:.4f}"
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index("Embedding model")


def observation_count_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        row = {
            "Embedding model": model_key,
            "Records": result["aggregate"]["record_count"],
        }
        for pair_key, _, _, label in PAIR_SPECS:
            row[label] = result["aggregate"]["pairs"][pair_key]["count"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("Embedding model")


def summarize_pairwise_preference(result: dict) -> dict:
    tolerance = 1e-12
    gaps = np.concatenate([
            (
                np.asarray(record["pair_values"]["p1-p2"], dtype=np.float64)[:, None]
                - np.asarray(record["pair_values"]["p1-p3"], dtype=np.float64)[None, :]
            ).reshape(-1)
            for record in result["records"]
    ])
    generality_preferred = int(np.count_nonzero(gaps > tolerance))
    locality_preferred = int(np.count_nonzero(gaps < -tolerance))
    ties = int(gaps.size - generality_preferred - locality_preferred)
    return {
        "comparisons": int(gaps.size),
        "mean_gap": float(gaps.mean()),
        "gap_std": float(gaps.std(ddof=0)),
        "generality_preferred": generality_preferred / gaps.size,
        "locality_preferred": locality_preferred / gaps.size,
        "ties": ties / gaps.size,
    }


def pairwise_preference_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        preference = result["pairwise_preference"]
        rows.append({
            "Embedding model": model_key,
            "Comparisons": preference["comparisons"],
            "Mean gap": preference["mean_gap"],
            "Gap std": preference["gap_std"],
            "Generality preferred": preference["generality_preferred"],
            "Locality preferred": preference["locality_preferred"],
            "Ties": preference["ties"],
        })
    return pd.DataFrame(rows).set_index("Embedding model")


def plot_similarity_bars(results: list[tuple[str, dict]], show_std: bool) -> None:
    model_keys = [model_key for model_key, _ in results]
    pair_specs = list(PAIR_SPECS)
    positions = np.arange(len(model_keys), dtype=float)
    group_width = 0.78
    bar_width = group_width / len(pair_specs)
    offsets = (
        np.arange(len(pair_specs), dtype=float) - (len(pair_specs) - 1) / 2
    ) * bar_width

    all_lower = []
    all_upper = []
    figure, axis = plt.subplots(figsize=(18, 7))
    for pair_offset, (pair_key, _, _, label) in zip(offsets, pair_specs):
        means = np.asarray([
            result["aggregate"]["pairs"][pair_key]["mean"]
            for _, result in results
        ])
        stds = np.asarray([
            result["aggregate"]["pairs"][pair_key]["std"]
            for _, result in results
        ])
        errors = stds if show_std else None
        axis.bar(
            positions + pair_offset,
            means,
            width=bar_width * 0.92,
            yerr=errors,
            label=f"{pair_key}: {label}",
            color=PAIR_COLORS[pair_key],
            capsize=5 if show_std else 0,
            error_kw={"elinewidth": 1.25, "capthick": 1.25},
        )
        all_lower.extend((means - stds if show_std else means).tolist())
        all_upper.extend((means + stds if show_std else means).tolist())

    y_min = min(0.0, min(all_lower))
    y_max = max(all_upper)
    padding = max(0.05, (y_max - y_min) * 0.12)
    axis.set_ylim(
        max(-1.0, y_min - (padding if y_min < 0 else 0)),
        min(1.0, y_max + padding),
    )
    axis.yaxis.set_major_locator(MaxNLocator(nbins=10))
    axis.set_xticks(positions, model_keys, rotation=20, ha="right")
    axis.set_ylabel("cosine similarity")
    record_counts = {result["aggregate"]["record_count"] for _, result in results}
    count_label = str(record_counts.pop()) if len(record_counts) == 1 else "mixed"
    axis.set_title(
        f"Sentence embedding similarity | n={count_label}",
        fontweight="bold",
    )
    axis.grid(axis="y", alpha=0.22)
    axis.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), title="prompt pair")
    figure.tight_layout(rect=(0, 0, 0.82, 1))
    plt.show()


def visualize_similarity(
    embedding_model_keys: list[str],
    record_limit: int | None,
    show_std: bool,
) -> None:
    model_keys = resolve_embedding_model_keys(embedding_model_keys)
    results = []
    for model_key in model_keys:
        output_path = similarity_output_path(model_key, record_limit)
        if not output_path.exists():
            raise FileNotFoundError(
                f"Embedding result does not exist: {output_path}. Calculate it first."
            )
        if not result_matches_configuration(output_path, model_key):
            raise ValueError(f"Stale embedding result: {output_path}. Recalculate it.")
        with open(output_path, encoding="utf-8") as file:
            result = json.load(file)
        result["pairwise_preference"] = summarize_pairwise_preference(result)
        result.pop("records", None)
        results.append((model_key, result))

    display(aggregate_pair_table(results, show_std=show_std).style)
    display(
        pairwise_preference_table(results).style.format({
            "Comparisons": "{:,}",
            "Mean gap": "{:.4f}",
            "Gap std": "{:.4f}",
            "Generality preferred": "{:.4f}",
            "Locality preferred": "{:.4f}",
            "Ties": "{:.4f}",
        })
    )
    display(
        observation_count_table(results).style.format("{:,}")
    )
    plot_similarity_bars(results, show_std=show_std)

## Behavior Configuration

Change these values immediately before selecting an action below.

In [ ]:
# Comment out a line to disable it; list order controls calculation and display order.
EMBEDDING_MODEL_KEYS = [
    "all-mpnet-base-v2",
    "all-MiniLM-L6-v2",
    "all-MiniLM-L12-v2",
    "e5-base-v2",
    "e5-large-v2",
    "bge-base-en-v1.5",
    "bge-large-en-v1.5",
    "bge-m3",
]

EMBEDDING_GPU_ID = 0
EMBEDDING_USE_FP16 = False
EMBEDDING_MAX_LENGTH = 384
EMBEDDING_BATCH_SIZE = 64
RECORD_BATCH_SIZE = 64  # Records are flattened, embedded in batches, then split.
RECORD_LIMIT = None  # Set a positive integer for diagnostic result files.
OVERWRITE_EXISTING_RESULT = False

PLOT_SHOW_STD = True

## Manual Actions

Uncomment one action at a time. Visualization reads only the existing JSON and does not load the dataset or embedding model.

In [ ]:
# Calculate and save one JSON per enabled embedding model:
# calculate_similarity(
#     embedding_model_keys=EMBEDDING_MODEL_KEYS,
#     record_limit=RECORD_LIMIT,
#     record_batch_size=RECORD_BATCH_SIZE,
#     overwrite_output=OVERWRITE_EXISTING_RESULT,
# )

# Visualize all enabled embedding-model results in list order:
# visualize_similarity(
#     embedding_model_keys=EMBEDDING_MODEL_KEYS,
#     record_limit=RECORD_LIMIT,
#     show_std=PLOT_SHOW_STD,
# )